<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자기지도 학습 · 멀티모달 · 08. BLIP: 이해·생성과 CapFilt

## BLIP: Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding and Generation

- **원 논문:** [BLIP: Bootstrapping Language-Image Pre-training for Unified Vision-Language Understanding and Generation](https://arxiv.org/abs/2201.12086)
- **저자 / 발표:** Junnan Li, Dongxu Li, Caiming Xiong, Steven Hoi · ICML (2022)
- **난이도 / 예상 시간:** 고급 · 약 85분
- **선수 지식:** CLIP, image-text matching, language modeling
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** MED의 ITC·ITM·LM 세 목적과 CapFilt의 생성/필터 단계를 작은 paired corpus에서 재현합니다.

**축소하거나 생략한 부분:** 1.29억 web pairs·ViT/BERT 대신 240개 pair와 작은 shared text embedding을 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1, Figure 2 | ITC·ITM·LM 세 동작 모드 | 세 loss 유한값 |
| §3.1 Image-Text Contrastive | unimodal alignment | 대칭 CE |
| §3.1 Image-Text Matching/LM | negative match·causal token prediction | binary/next-token loss |
| §3.2, Figure 3 | captioner 생성+filter 제거 CapFilt | 정제 전후 noise |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=random.seed,np.random.seed,torch.manual_seed,torch.cuda.manual_seed_all,Path,np.load,torch.tensor,torch.float32 -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 8개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>random.seed(a=None)</code></summary>

#### `random.seed(a=None)`

- **역할:** Python 표준 난수 생성기의 상태를 초기화합니다.
- **입력·반환:** 정수 등의 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 같은 Python 난수 수열을 다시 만들어 실험을 재현합니다.
- **주의점:** NumPy와 PyTorch의 난수 상태는 별도로 고정해야 합니다.
- **공식 문서:** [random.seed](https://docs.python.org/3/library/random.html#random.seed)

</details>

<details>
<summary><code>np.random.seed(seed=None)</code></summary>

#### `np.random.seed(seed=None)`

- **역할:** NumPy의 기존 전역 난수 생성기 상태를 초기화합니다.
- **입력·반환:** 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 레거시 `np.random.*` 호출의 수열을 재현합니다.
- **주의점:** 새 코드에서는 독립 상태를 가진 `default_rng`가 권장됩니다.
- **공식 문서:** [np.random.seed](https://numpy.org/doc/stable/reference/random/generated/numpy.random.seed.html)

</details>

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>torch.cuda.manual_seed_all(seed)</code></summary>

#### `torch.cuda.manual_seed_all(seed)`

- **역할:** 사용 가능한 모든 CUDA GPU의 난수 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 여러 GPU에서도 초기화와 샘플링을 최대한 반복 가능하게 합니다.
- **주의점:** CPU 시드와 결정론 설정을 대신하지 않습니다.
- **공식 문서:** [torch.cuda.manual_seed_all](https://docs.pytorch.org/docs/stable/generated/torch.cuda.manual_seed_all.html)

</details>

<details>
<summary><code>Path(*pathsegments)</code></summary>

#### `Path(*pathsegments)`

- **역할:** 운영체제에 맞는 파일 경로 객체를 만듭니다.
- **입력·반환:** 경로 조각을 받고 `Path` 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** Windows에서도 문자열 결합 없이 데이터·결과 경로를 구성합니다.
- **주의점:** 상대 경로는 현재 작업 디렉터리를 기준으로 해석됩니다.
- **공식 문서:** [Path](https://docs.python.org/3/library/pathlib.html#pathlib.Path)

</details>

<details>
<summary><code>np.load(file, allow_pickle=False, ...)</code></summary>

#### `np.load(file, allow_pickle=False, ...)`

- **역할:** NumPy 배열이나 archive를 파일에서 읽습니다.
- **입력·반환:** 배열과 축·dtype 등의 설정을 받고 배열 또는 수치 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** 데이터 shape과 수치 연산을 논문 실험에 맞게 구성·검증합니다.
- **주의점:** 축, dtype, 복사·view 여부와 빈 배열에서의 동작을 확인해야 합니다.
- **공식 문서:** [np.load](https://numpy.org/doc/stable/reference/generated/numpy.load.html)

</details>

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.float32</code></summary>

#### `torch.float32`

- **역할:** 32비트 부동소수점 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** 일반적인 학습 실수 dtype입니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.float32](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$\mathcal L_{BLIP}=\mathcal L_{ITC}+\mathcal L_{ITM}+\mathcal L_{LM}$$

- **기호 정의:** ITC는 정렬, ITM은 pair 일치, LM은 다음 token 생성 목적입니다.
- **수식의 역할:** 하나의 모델에 이해와 생성을 결합하고 생성·필터 과정으로 caption을 정제합니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 §3.1, Figure 2을 구현합니다.
- **입출력 shape:** image [B, 1, 8, 8], token [B, L] → contrastive [B, B], LM [B, L-1, V]
- **평가:** 세 loss, retrieval, clean/noisy pair filtering precision
- **원문 대비 한계:** 1.29억 web pairs·ViT/BERT 대신 240개 pair와 작은 shared text embedding을 사용합니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

### 구현 단계 2 · 합성 입력·데이터 계약 (data-contract 예외)

- **원문 위치:** 해당 없음 — 원 논문의 학습 연산이 아니라 외부 다운로드를 없애기 위한 축소 입력 생성 단계입니다.
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 논문 메커니즘과 데이터 규모 효과를 분리하고 Windows CPU에서도 같은 입력을 재현합니다.
- **완료 증거:** pair 정렬, batch 크기, dtype과 입력 rank assertion이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.stack,torch.where -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.stack(tensors, dim=0)</code></summary>

#### `torch.stack(tensors, dim=0)`

- **역할:** 새 축을 만들어 텐서를 쌓습니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{shape}(Y)=(d_1,\ldots,d_{a-1},R,d_a,\ldots,d_K)
$$

- **기호:** $R$은 쌓는 tensor 개수이고 $a$는 새 축이 들어갈 위치입니다.
- **수식 → 코드:** `stack`은 각 입력을 새 index 하나로 배치해 새로운 축을 만듭니다.
- **직관:** 연결할 기존 축을 고르는 `cat`과 달리 축의 개수가 하나 늘어납니다.
- **shape 확인:** 모든 입력 shape이 완전히 같아야 합니다.
- **공식 문서:** [torch.stack](https://docs.pytorch.org/docs/stable/generated/torch.stack.html)

</details>

<details>
<summary><code>torch.where(condition, input, other)</code></summary>

#### `torch.where(condition, input, other)`

- **역할:** 조건에 따라 두 텐서의 원소를 선택합니다.
- **입력·반환:** 불리언 조건과 두 입력을 받고 broadcast된 결과를 반환합니다.
- **이 셀에서 쓰는 이유:** mask 기반 분기와 piecewise 수식을 벡터화합니다.
- **주의점:** 세 입력의 broadcasting과 두 branch의 dtype·device를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y_i=\begin{cases}x_i,&c_i=\mathrm{True}\\z_i,&c_i=\mathrm{False}\end{cases}
$$

- **기호:** $c_i$는 조건, $x_i,z_i$는 두 후보 값입니다.
- **수식 → 코드:** API가 Python 분기 대신 원소별로 두 tensor 중 하나를 선택합니다.
- **직관:** piecewise 수식과 mask 기반 update를 병렬 tensor 연산으로 표현합니다.
- **shape 확인:** 세 입력을 broadcasting한 공통 shape이 출력 shape입니다.
- **공식 문서:** [torch.where](https://docs.pytorch.org/docs/stable/generated/torch.where.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
image_features = torch.tensor(raw["image_features"], dtype=torch.float32)
class_image_features = torch.tensor(raw["class_image_features"], dtype=torch.float32)
class_tokens = torch.stack(
    [tokens[torch.where(class_ids == c)[0][0]] for c in class_ids.unique(sorted=True)]
)

### 구현 단계 3 · 합성 입력·데이터 계약 (data-contract 예외)

- **원문 위치:** 해당 없음 — 원 논문의 학습 연산이 아니라 외부 다운로드를 없애기 위한 축소 입력 생성 단계입니다.
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 논문 메커니즘과 데이터 규모 효과를 분리하고 Windows CPU에서도 같은 입력을 재현합니다.
- **완료 증거:** pair 정렬, batch 크기, dtype과 입력 rank assertion이 통과해야 합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.arange,F.normalize -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

<details>
<summary><code>F.normalize(input, p=2.0, dim=1, eps=1e-12)</code></summary>

#### `F.normalize(input, p=2.0, dim=1, eps=1e-12)`

- **역할:** 지정 축의 벡터 norm을 1로 정규화합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\hat x=\frac{x}{\max(\lVert x\rVert_p,\epsilon)}
$$

- **기호:** $p$는 norm 차수, $\epsilon$은 0으로 나누지 않기 위한 하한입니다.
- **수식 → 코드:** 코드의 `dim` 축에서 norm을 구한 뒤 모든 원소를 같은 값으로 나눕니다.
- **직관:** 벡터 크기를 제거해 방향이나 상대 패턴에 집중하게 합니다.
- **shape 확인:** 입력과 출력 shape은 같고 선택 축의 norm은 약 1입니다.
- **공식 문서:** [F.normalize](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.normalize.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def make_web_and_synthetic(text, image_feat, rate=0.35):
    web = text.clone()
    count = int(len(text) * rate)
    bad = torch.arange(count)
    web[bad] = web[bad.roll(1)]
    nearest = (
        F.normalize(image_feat, dim=1) @ F.normalize(class_image_features, dim=1).T
    ).argmax(1)
    synthetic = class_tokens[nearest]
    return web, synthetic, bad


web, synthetic, bad = make_web_and_synthetic(tokens, image_features)
assert len(bad) == int(len(tokens) * 0.35) and synthetic.shape == tokens.shape

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §3.2, Figure 3
- **이 셀의 책임:** captioner 생성+filter 제거 CapFilt
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 정제 전후 noise. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.bool -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.bool</code></summary>

#### `torch.bool`

- **역할:** 불리언 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** mask와 조건 결과를 표현합니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.bool](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def capfilt(web: Tensor, synthetic: Tensor):
    keep = (web == synthetic).float().mean(1) >= 0.5
    cleaned = web.clone()
    cleaned[~keep] = synthetic[~keep]
    return cleaned, keep


cleaned, keep = capfilt(web, synthetic)
before = float((web != tokens).any(1).float().mean())
after = float((cleaned != tokens).any(1).float().mean())
assert keep.dtype == torch.bool and after < before

### 구현 단계 5 · 핵심 연산·모델

- **원문 위치:** §3.1, Figure 2 / §3.1 Image-Text Contrastive / §3.1 Image-Text Matching/LM
- **이 셀의 책임:** ITC·ITM·LM 세 동작 모드 / unimodal alignment / negative match·causal token prediction
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 세 loss 유한값 / 대칭 CE / binary/next-token loss. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Sequential,nn.Linear,nn.Embedding,F.cross_entropy,torch.cat,torch.ones,torch.zeros,F.binary_cross_entropy_with_logits,Tensor.reshape,torch.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 10개 · 수식 6개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>nn.Embedding(num_embeddings, embedding_dim, ...)</code></summary>

#### `nn.Embedding(num_embeddings, embedding_dim, ...)`

- **역할:** 정수 ID를 학습 가능한 벡터로 조회합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
e_i=E[i],\qquad E\in\mathbb{R}^{V\times D}
$$

- **기호:** $i$는 정수 ID, $V$는 vocabulary 크기, $D$는 embedding 차원입니다.
- **수식 → 코드:** forward는 one-hot 행렬곱을 만들지 않고 weight 행 $E[i]$를 직접 조회합니다.
- **직관:** 범주 ID를 학습 가능한 연속 벡터 좌표로 바꿉니다.
- **shape 확인:** 입력 `[...] -> 출력 [..., D]`; 입력 dtype은 보통 `torch.int64`입니다.
- **공식 문서:** [nn.Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

<details>
<summary><code>torch.cat(tensors, dim=0)</code></summary>

#### `torch.cat(tensors, dim=0)`

- **역할:** 기존 축을 따라 텐서를 연결합니다.
- **입력·반환:** 텐서와 축·연산 설정을 받고 변환된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 수식의 축 관계와 batch 연산을 코드로 직접 표현합니다.
- **주의점:** 입력 shape과 broadcasting 규칙을 셀의 shape assertion으로 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
d_{\mathrm{out}}^{(a)}=\sum_{r=1}^{R}d_r^{(a)}
$$

- **기호:** $a$는 연결할 축이고 $R$은 입력 tensor 개수입니다.
- **수식 → 코드:** `dim=a` 축의 길이만 더하고 나머지 축은 그대로 유지합니다.
- **직관:** 이미 존재하는 축을 길게 이어 붙이는 연산입니다.
- **shape 확인:** 연결 축을 제외한 모든 shape과 dtype·device가 같아야 합니다.
- **공식 문서:** [torch.cat](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

</details>

<details>
<summary><code>torch.ones(*size, dtype=None, device=None)</code></summary>

#### `torch.ones(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 1인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 weight·mask·스케일을 구성합니다.
- **주의점:** 주변 연산과 dtype·device가 같은지 확인해야 합니다.
- **공식 문서:** [torch.ones](https://docs.pytorch.org/docs/stable/generated/torch.ones.html)

</details>

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

<details>
<summary><code>F.binary_cross_entropy_with_logits(input, target, ...)</code></summary>

#### `F.binary_cross_entropy_with_logits(input, target, ...)`

- **역할:** sigmoid와 binary cross entropy를 안정적으로 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{BCE}=-\frac{1}{N}\sum_i\left[y_i\log\sigma(z_i)+(1-y_i)\log(1-\sigma(z_i))\right]
$$

- **기호:** $z_i$는 logit, $y_i\in[0,1]$은 target, $\sigma$는 sigmoid입니다.
- **수식 → 코드:** sigmoid와 BCE를 한 함수에서 log-sum-exp 형태로 안정적으로 계산합니다.
- **직관:** 각 항을 독립적인 이진 선택으로 보고 정답 쪽 확률의 음의 로그를 최소화합니다.
- **shape 확인:** logit과 target은 같거나 broadcast 가능한 shape이어야 합니다.
- **공식 문서:** [F.binary_cross_entropy_with_logits](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.binary_cross_entropy_with_logits.html)

</details>

<details>
<summary><code>tensor.reshape(*shape)</code></summary>

#### `tensor.reshape(*shape)`

- **역할:** 원소 수를 유지하며 텐서 shape을 바꿉니다.
- **입력·반환:** 새 shape을 받고 가능하면 view, 아니면 복사된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch·head·sequence 축을 논문 수식의 shape에 맞춥니다.
- **주의점:** 원소 수가 같아야 하며 메모리 공유 여부에 의존하면 안 됩니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [Tensor.reshape](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)

</details>

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class TinyBLIP(nn.Module):
    def __init__(self, vocab, dim=16):
        super().__init__()
        self.vocab = vocab
        self.image = nn.Sequential(nn.Flatten(), nn.Linear(64, dim))
        self.word = nn.Embedding(vocab, dim)
        self.itm = nn.Linear(2 * dim, 1)
        self.lm = nn.Linear(2 * dim, vocab)

    def losses(self, x, t):
        zi = F.normalize(self.image(x), dim=1)
        zt = F.normalize(self.word(t).mean(1), dim=1)
        target = torch.arange(len(x), device=x.device)
        sim = zi @ zt.T / 0.15
        itc = (F.cross_entropy(sim, target) + F.cross_entropy(sim.T, target)) / 2
        neg = zt.roll(1, 0)
        match_logits = torch.cat(
            [self.itm(torch.cat([zi, zt], 1)), self.itm(torch.cat([zi, neg], 1))]
        )
        itm_target = torch.cat(
            [
                torch.ones(len(x), 1, device=x.device),
                torch.zeros(len(x), 1, device=x.device),
            ]
        )
        itm = F.binary_cross_entropy_with_logits(match_logits, itm_target)
        context = zi[:, None, :].expand(-1, t.shape[1] - 1, -1)
        lm_logits = self.lm(torch.cat([self.word(t[:, :-1]), context], 2))
        lm = F.cross_entropy(lm_logits.reshape(-1, self.vocab), t[:, 1:].reshape(-1))
        return itc, itm, lm, sim


blip = TinyBLIP(int(tokens.max()) + 1)
parts = blip.losses(images[:24], cleaned[:24])
assert all(torch.isfinite(v) for v in parts[:3])

### 구현 단계 6 · 평가·완료 증거

- **원문 위치:** §3.1, Figure 2 / §3.1 Image-Text Contrastive / §3.1 Image-Text Matching/LM / §3.2, Figure 3
- **이 셀의 책임:** ITC·ITM·LM 세 동작 모드 / unimodal alignment / negative match·causal token prediction / captioner 생성+filter 제거 CapFilt
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 세 loss 유한값 / 대칭 CE / binary/next-token loss / 정제 전후 noise. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=ACCELERATOR.move,torch.optim.Adam,Module.parameters,Optimizer.zero_grad,Tensor.backward,Optimizer.step,Tensor.detach,torch.no_grad,Tensor.argmax -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 9개 · 수식 4개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>ACCELERATOR.move(value)</code></summary>

#### `ACCELERATOR.move(value)`

- **역할:** 텐서나 모듈을 선택된 실행 장치로 옮기는 프로젝트 도우미입니다.
- **입력·반환:** 이동할 값을 받고 같은 논리 값을 장치에 배치해 반환합니다.
- **이 셀에서 쓰는 이유:** 모델과 batch가 서로 다른 장치에 놓이는 오류를 막습니다.
- **주의점:** 컨테이너 재귀 처리 여부와 dtype 변경 여부는 프로젝트 구현을 확인해야 합니다.
- **공식 문서:** 프로젝트 내부 도우미이므로 정의의 docstring을 확인합니다.

</details>

<details>
<summary><code>torch.optim.Adam(params, lr=1e-3, ...)</code></summary>

#### `torch.optim.Adam(params, lr=1e-3, ...)`

- **역할:** 1·2차 gradient 모멘트를 추적하는 Adam optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 노트북의 trainable parameter를 반복 update합니다.
- **주의점:** weight decay의 해석은 AdamW와 다르며 학습률이 결과에 크게 작용합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}m_t&=\beta_1m_{t-1}+(1-\beta_1)g_t,\\v_t&=\beta_2v_{t-1}+(1-\beta_2)g_t^2,\\\theta_t&=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}\end{aligned}
$$

- **기호:** $g_t$는 gradient, $m_t,v_t$는 1·2차 모멘트, $\eta$는 학습률입니다.
- **수식 → 코드:** `backward()`가 만든 gradient를 읽어 모멘트를 갱신한 뒤 `step()`에서 파라미터에 적용합니다.
- **직관:** 최근 gradient 방향과 크기를 함께 누적해 파라미터별 적응형 보폭을 만듭니다.
- **shape 확인:** 각 trainable parameter와 같은 shape의 모멘트 상태가 optimizer에 저장됩니다.
- **공식 문서:** [torch.optim.Adam](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

<details>
<summary><code>tensor.argmax(dim=None, keepdim=False)</code></summary>

#### `tensor.argmax(dim=None, keepdim=False)`

- **역할:** 지정 축에서 최댓값의 index를 찾습니다.
- **입력·반환:** 텐서와 축을 받고 정수 index 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** logit을 예측 class나 선택 action으로 바꿉니다.
- **주의점:** 미분 불가능하며 동점에서는 첫 index가 선택됩니다.

##### 관련 수식과 코드 연결

$$
k^\star=\operatorname*{arg\,max}_{k}x_k
$$

- **기호:** $k^\star$는 가장 큰 값 자체가 아니라 그 값의 index입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 어느 후보 집합에서 최댓값 index를 고를지 정합니다.
- **직관:** 연속 score를 이산 class·action 결정으로 바꾸는 선택 연산입니다.
- **shape 확인:** 선택 축이 제거되며 미분 가능한 gradient 경로는 만들지 않습니다.
- **공식 문서:** [Tensor.argmax](https://docs.pytorch.org/docs/stable/generated/torch.argmax.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
blip = ACCELERATOR.move(blip)
idx = train_idx[:96]
train_images, train_cleaned = ACCELERATOR.move(images[idx], cleaned[idx])
optimizer = torch.optim.Adam(blip.parameters(), lr=0.012)
losses = []
for _ in range(130):
    itc, itm, lm, _ = blip.losses(train_images, train_cleaned)
    loss = itc + itm + 0.5 * lm
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
with torch.no_grad():
    test_images, test_cleaned = ACCELERATOR.move(images[test_idx], cleaned[test_idx])
    *_, sim = blip.losses(test_images, test_cleaned)
    retrieval = float(
        (sim.argmax(1) == torch.arange(len(test_idx), device=sim.device)).float().mean()
    )
assert losses[-1] < losses[0] and after < before
fig, ax = plt.subplots(1, 2, figsize=(8, 3))
ax[0].plot(losses)
ax[0].set_title("ITC+ITM+LM")
ax[1].bar(["web", "CapFilt"], [before, after])
ax[1].set_title("caption noise rate")
plt.tight_layout()
plt.show()
print(
    {"noise_before": before, "noise_after": after, "retrieval@1": round(retrieval, 3)}
)

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 7 · 핵심 연산·모델

- **원문 위치:** §3.1, Figure 2 / §3.1 Image-Text Contrastive / §3.1 Image-Text Matching/LM
- **이 셀의 책임:** ITC·ITM·LM 세 동작 모드 / unimodal alignment / negative match·causal token prediction
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 세 loss 유한값 / 대칭 CE / binary/next-token loss. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=dataclass,nn.GRU,Tensor.mean -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>@dataclass(frozen=False, slots=False, ...)</code></summary>

#### `@dataclass(frozen=False, slots=False, ...)`

- **역할:** type annotation을 바탕으로 초기화·표현 메서드 등을 생성합니다.
- **입력·반환:** class decorator 설정을 받고 변환된 class를 반환합니다.
- **이 셀에서 쓰는 이유:** 실험 설정과 결과 record의 필드를 명시적으로 정의합니다.
- **주의점:** mutable 기본값은 `default_factory`를 쓰고 equality·frozen 의미를 확인합니다.
- **공식 문서:** [dataclass](https://docs.python.org/3/library/dataclasses.html#dataclasses.dataclass)

</details>

<details>
<summary><code>nn.GRU(input_size, hidden_size, ...)</code></summary>

#### `nn.GRU(input_size, hidden_size, ...)`

- **역할:** gated recurrent unit 층을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\begin{aligned}r_t&=\sigma(W_{ir}x_t+b_{ir}+W_{hr}h_{t-1}+b_{hr}),\\z_t&=\sigma(W_{iz}x_t+b_{iz}+W_{hz}h_{t-1}+b_{hz}),\\n_t&=\tanh(W_{in}x_t+b_{in}+r_t\odot(W_{hn}h_{t-1}+b_{hn})),\\h_t&=(1-z_t)\odot n_t+z_t\odot h_{t-1}\end{aligned}
$$

- **기호:** $r_t,z_t$는 reset·update gate, $n_t$는 후보, $h_t$는 hidden state입니다.
- **수식 → 코드:** `nn.GRU`가 time step마다 이 네 식을 계산하고 hidden state를 다음 step으로 넘깁니다.
- **직관:** gate가 과거 정보를 얼마나 지우고 유지할지 학습합니다.
- **shape 확인:** `batch_first=True`이면 입력 `[B,T,D]`, 출력 `[B,T,H]`, 마지막 hidden `[L,B,H]`입니다.
- **공식 문서:** [nn.GRU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html)

</details>

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 16
    learning_rate: float = 0.02
    steps: int = 14


def prepare_views(image_batch: Tensor, text_batch: Tensor) -> tuple[Tensor, Tensor]:
    return image_batch, text_batch


class BlipPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.vocabulary = int(tokens.max()) + 1
        self.image_encoder = nn.Sequential(
            nn.Flatten(), nn.Linear(64, config.hidden_dim)
        )
        self.token_embedding = nn.Embedding(self.vocabulary, config.hidden_dim)
        self.matching_head = nn.Linear(config.hidden_dim * 2, 1)
        self.language_model = nn.GRU(
            config.hidden_dim, config.hidden_dim, batch_first=True
        )
        self.token_head = nn.Linear(config.hidden_dim, self.vocabulary)

    def forward(self, image_batch: Tensor, text_batch: Tensor) -> dict[str, Tensor]:
        image_embedding = self.image_encoder(image_batch)
        token_hidden = self.token_embedding(text_batch)
        text_embedding = token_hidden.mean(dim=1)
        contrastive = F.normalize(image_embedding, dim=1)
        contrastive = contrastive @ F.normalize(text_embedding, dim=1).T
        positive_pair = torch.cat([image_embedding, text_embedding], dim=1)
        negative_pair = torch.cat(
            [image_embedding, text_embedding.roll(shifts=1, dims=0)], dim=1
        )
        matching = self.matching_head(torch.cat([positive_pair, negative_pair], dim=0))
        # BLIP §3.1 LM mode: image representation이 decoder initial state를
        # condition하므로 caption token 예측이 image에 따라 달라집니다.
        lm_hidden, _ = self.language_model(
            token_hidden[:, :-1], image_embedding.unsqueeze(0)
        )
        token_logits = self.token_head(lm_hidden)
        return {"contrastive": contrastive, "matching": matching, "lm": token_logits}

    def compute_loss(self, outputs: dict[str, Tensor], text_batch: Tensor) -> Tensor:
        target = torch.arange(len(text_batch), device=text_batch.device)
        contrastive = 0.5 * (
            F.cross_entropy(outputs["contrastive"], target)
            + F.cross_entropy(outputs["contrastive"].T, target)
        )
        matching = F.binary_cross_entropy_with_logits(
            outputs["matching"],
            torch.cat(
                [
                    torch.ones(len(text_batch), 1, device=text_batch.device),
                    torch.zeros(len(text_batch), 1, device=text_batch.device),
                ],
                dim=0,
            ),
        )
        language = F.cross_entropy(
            outputs["lm"].reshape(-1, self.vocabulary),
            text_batch[:, 1:].reshape(-1),
        )
        return contrastive + matching + language

    def training_step(self, image_batch: Tensor, text_batch: Tensor) -> Tensor:
        return self.compute_loss(self(image_batch, text_batch), text_batch)

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 8 · 목적함수·학습 update

- **원문 위치:** §3.1, Figure 2
- **이 셀의 책임:** ITC·ITM·LM 세 동작 모드
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** 세 loss 유한값. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(
    model: BlipPortfolioModel,
    image_batch: Tensor,
    text_batch: Tensor,
) -> list[float]:
    optimizer = torch.optim.Adam(model.parameters(), lr=model.config.learning_rate)
    history = []
    for _ in range(model.config.steps):
        loss = model.training_step(image_batch, text_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return history

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 9 · 평가·완료 증거

- **원문 위치:** §3.1 Image-Text Contrastive / §3.1 Image-Text Matching/LM / §3.2, Figure 3
- **이 셀의 책임:** unimodal alignment / negative match·causal token prediction / captioner 생성+filter 제거 CapFilt
- **Tensor shape 계약:** image/text [B,*] → filtered pairs [B',*] → three objective scalars
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** 대칭 CE / binary/next-token loss / 정제 전후 noise. 코드의 assertion과 출력으로 바로 확인합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.sum,Tensor.cpu,np.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

<details>
<summary><code>tensor.cpu()</code></summary>

#### `tensor.cpu()`

- **역할:** 텐서를 CPU 메모리로 옮깁니다.
- **입력·반환:** 텐서를 받아 CPU 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** NumPy 변환이나 CPU 기반 평가 도구에 값을 전달합니다.
- **주의점:** GPU 텐서 이동은 동기화·복사 비용을 만들 수 있습니다.
- **공식 문서:** [Tensor.cpu](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.cpu.html)

</details>

<details>
<summary><code>np.isfinite(x)</code></summary>

#### `np.isfinite(x)`

- **역할:** 각 값이 NaN이나 무한대가 아닌 유한수인지 검사합니다.
- **입력·반환:** 배열형 입력을 받고 같은 shape의 불리언 배열을 반환합니다.
- **이 셀에서 쓰는 이유:** 학습 값이나 전처리 결과가 수치적으로 정상인지 검증합니다.
- **주의점:** 유한하다는 사실만 확인하며 값의 크기가 적절하다는 뜻은 아닙니다.
- **공식 문서:** [np.isfinite](https://numpy.org/doc/stable/reference/generated/numpy.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def evaluate_portfolio_model(
    model: BlipPortfolioModel,
    image_batch: Tensor,
    text_batch: Tensor,
    keep_mask: Tensor,
) -> dict[str, float]:
    with torch.no_grad():
        outputs = model(image_batch, text_batch)
        similarity = outputs["contrastive"]
        target = torch.arange(len(similarity), device=similarity.device)
        recall = (similarity.argmax(1) == target).float().mean()
        rolled_outputs = model(image_batch.roll(shifts=1, dims=0), text_batch)
        conditioning_gap = (outputs["lm"] - rolled_outputs["lm"]).abs().mean()
        positive_logit, negative_logit = outputs["matching"].chunk(2, dim=0)
        matching_margin = positive_logit.mean() - negative_logit.mean()
        retained_count = int(keep_mask.sum().item())
        replaced_count = int((~keep_mask).sum().item())
    return {
        "retrieval_recall1": float(recall.cpu()),
        "lm_image_conditioning_gap": float(conditioning_gap.cpu()),
        "itm_positive_negative_margin": float(matching_margin.cpu()),
        "capfilt_retained_count": float(retained_count),
        "capfilt_replaced_count": float(replaced_count),
    }


portfolio_train_index = train_idx[:96]
portfolio_test_index = test_idx
assert not torch.isin(portfolio_train_index, portfolio_test_index).any()
portfolio_train_images, portfolio_cleaned_tokens = prepare_views(
    images[portfolio_train_index],
    cleaned[portfolio_train_index],
)
portfolio_train_images, portfolio_cleaned_tokens = ACCELERATOR.move(
    portfolio_train_images,
    portfolio_cleaned_tokens,
)
(
    portfolio_test_images,
    portfolio_test_cleaned_tokens,
    portfolio_test_keep,
) = ACCELERATOR.move(
    images[portfolio_test_index],
    cleaned[portfolio_test_index],
    keep[portfolio_test_index],
)
portfolio_model = ACCELERATOR.move(BlipPortfolioModel(PortfolioConfig()))
portfolio_history = fit_portfolio_model(
    portfolio_model,
    portfolio_train_images,
    portfolio_cleaned_tokens,
)
portfolio_metrics = evaluate_portfolio_model(
    portfolio_model,
    portfolio_test_images,
    portfolio_test_cleaned_tokens,
    portfolio_test_keep,
)
assert np.isfinite(portfolio_history).all()
assert portfolio_metrics["lm_image_conditioning_gap"] > 0.0
assert portfolio_metrics["capfilt_retained_count"] + portfolio_metrics[
    "capfilt_replaced_count"
] == len(portfolio_test_index)
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> 1.29억 web pairs·ViT/BERT 대신 240개 pair와 작은 shared text embedding을 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.